NOTEBOOK 2: 06_explainability_FIXED.ipynb<br>
Explainability & Interpretability for CNN-LSTM


<br>
Updated for:<br>
- Single CNN-LSTM model (removed multi-model logic)<br>
- Proper gradient computation<br>
- Enhanced visualizations<br>
- Better integration with trained model<br>


Cell 1: Setup and Imports

In [ ]:
import os
import pickle
import warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm
from scipy.ndimage import gaussian_filter1d
import torch
import torch.nn as nn
import torch.nn.functional as F
warnings.filterwarnings('ignore')

Cell 2: Paths and Configuration

In [ ]:
SAVE_DIR = os.path.join('..', 'data', 'processed')
MODEL_DIR = os.path.join('..', 'reports', 'checkpoints')
FIG_DIR = os.path.join('..', 'reports', 'figures', 'explainability')
os.makedirs(FIG_DIR, exist_ok=True)

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
FS = 100
INPUT_LEN = 500
HORIZON = 100
N_LEADS = 12
LEAD_NAMES = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']

Cell 3: Theme Configuration

In [ ]:
COLORS = {
    'bg': '#ffffff',
    'grid': '#e0e0e0',
    'text': '#2c3e50',
    'accent1': '#3498db',
    'accent2': '#e74c3c',
    'accent3': '#2ecc71',
    'accent4': '#f39c12',
}

In [ ]:
plt.rcParams.update({
    'figure.facecolor': COLORS['bg'],
    'axes.facecolor': COLORS['bg'],
    'axes.grid': True,
    'grid.alpha': 0.3,
    'figure.dpi': 120,
    'savefig.dpi': 150,
})

In [ ]:
def save_fig(name):
    path = os.path.join(FIG_DIR, name)
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved → {name}")

In [ ]:
print(f"Device: {DEVICE}")
print(f"Input: {INPUT_LEN/FS:.1f}s, Horizon: {HORIZON/FS:.1f}s")

Cell 4: Load Data

In [ ]:
X_test = np.load(os.path.join(SAVE_DIR, 'X_test.npy'))
y_test = np.load(os.path.join(SAVE_DIR, 'y_test.npy'))
config = pickle.load(open(os.path.join(SAVE_DIR, 'config.pkl'), 'rb'))

In [ ]:
n_samples_explain = min(50, len(X_test))
X_explain = X_test[:n_samples_explain]
y_explain = y_test[:n_samples_explain]

In [ ]:
X_explain_torch = torch.tensor(X_explain, dtype=torch.float32).to(DEVICE)
y_explain_torch = torch.tensor(y_explain, dtype=torch.float32).to(DEVICE)

In [ ]:
print(f"X_explain shape: {X_explain.shape}")

Cell 5: CNN-LSTM Model Definition

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k, pool=True):
        super().__init__()
        ops = [nn.Conv1d(in_ch, out_ch, kernel_size=k, padding=k//2, bias=False),
               nn.BatchNorm1d(out_ch), nn.GELU()]
        if pool:
            ops.append(nn.MaxPool1d(2))
        self.net = nn.Sequential(*ops)
    def forward(self, x):
        return self.net(x)

In [ ]:
class CNNLSTMForecaster(nn.Module):
    def __init__(self, n_leads=12, horizon=100, dropout=0.2, lstm_hidden=128, bidirectional=True):
        super().__init__()
        self.horizon = horizon
        self.n_leads = n_leads
        self.D = 2 if bidirectional else 1
        self.cnn = nn.Sequential(
            ConvBlock(n_leads, 32, k=7, pool=True),
            ConvBlock(32, 64, k=5, pool=True),
            ConvBlock(64, 128, k=3, pool=True),
            ConvBlock(128, 128, k=3, pool=False),
        )
        self.lstm = nn.LSTM(input_size=128, hidden_size=lstm_hidden, num_layers=2,
                           batch_first=True, dropout=dropout, bidirectional=bidirectional)
        self.attn = nn.Sequential(nn.Linear(lstm_hidden * self.D, 64),
                                  nn.Tanh(), nn.Linear(64, 1))
        self.decoder = nn.Sequential(
            nn.LayerNorm(lstm_hidden * self.D),
            nn.Dropout(dropout),
            nn.Linear(lstm_hidden * self.D, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, horizon * n_leads),
        )
    def forward(self, x):
        f = self.cnn(x).permute(0, 2, 1)
        enc, _ = self.lstm(f)
        w = torch.softmax(self.attn(enc), dim=1)
        ctx = (w * enc).sum(dim=1)
        out = self.decoder(ctx)
        return out.view(-1, self.horizon, self.n_leads)

Cell 6: Load Trained Model

In [ ]:
model = CNNLSTMForecaster(n_leads=N_LEADS, horizon=HORIZON, dropout=0.2).to(DEVICE)

In [ ]:
model_path = os.path.join(MODEL_DIR, 'CNN-LSTM_final.pt')
if os.path.exists(model_path):
    checkpoint = torch.load(model_path, map_location=DEVICE)
    if isinstance(checkpoint, dict):
        state_dict = checkpoint.get('model_state_dict', checkpoint)
    else:
        state_dict = checkpoint
    model.load_state_dict(state_dict)
    print("✅ Model loaded successfully")
else:
    print("⚠️ No checkpoint found, using untrained model")

In [ ]:
model.eval()

Cell 7: Gradient-Based Attribution

In [ ]:
def compute_gradient_attribution(model, X_batch, y_batch, device):
    X_batch = X_batch.clone().detach().requires_grad_(True)
    y_pred = model(X_batch)
    loss = F.mse_loss(y_pred, y_batch)
    loss.backward()
    attribution = X_batch.grad * X_batch.detach()
    return attribution.detach(), y_pred.detach()

In [ ]:
print("Computing gradient-based attributions...")
attributions, predictions = compute_gradient_attribution(model, X_explain_torch, y_explain_torch, DEVICE)

In [ ]:
attributions_np = attributions.cpu().numpy()
predictions_np = predictions.cpu().numpy()
print(f"Attributions shape: {attributions_np.shape}")

Cell 8: Attribution Heatmaps

In [ ]:
n_viz = 3
fig = plt.figure(figsize=(16, 3.5*n_viz))

In [ ]:
for idx in range(n_viz):
    ax1 = plt.subplot(n_viz, 2, 2*idx + 1)
    attr_sample = attributions_np[idx]
    norm = TwoSlopeNorm(vmin=attr_sample.min(), vcenter=0, vmax=attr_sample.max())
    im1 = ax1.imshow(attr_sample.T, aspect='auto', cmap='RdBu_r', norm=norm)
    ax1.set_ylabel('Lead')
    ax1.set_yticks(range(N_LEADS))
    ax1.set_yticklabels(LEAD_NAMES, fontsize=8)
    ax1.set_xlabel('Time (samples)')
    ax1.set_title(f'Sample {idx+1}: Gradient×Input Attribution', fontweight='bold')
    plt.colorbar(im1, ax=ax1, label='Attribution')
    
    ax2 = plt.subplot(n_viz, 2, 2*idx + 2)
    pred_error = predictions_np[idx] - y_explain[idx]
    norm2 = TwoSlopeNorm(vmin=pred_error.min(), vcenter=0, vmax=pred_error.max())
    im2 = ax2.imshow(pred_error.T, aspect='auto', cmap='RdBu_r', norm=norm2)
    ax2.set_ylabel('Lead')
    ax2.set_yticks(range(N_LEADS))
    ax2.set_yticklabels(LEAD_NAMES, fontsize=8)
    ax2.set_xlabel('Time (samples)')
    ax2.set_title(f'Sample {idx+1}: Prediction Error', fontweight='bold')
    plt.colorbar(im2, ax=ax2, label='Error')

In [ ]:
plt.suptitle('Gradient-Based Attribution & Prediction Error', fontsize=14, fontweight='bold')
plt.tight_layout()
save_fig('01_gradient_attribution_heatmaps.png')

Cell 9: Temporal & Lead Importance

In [ ]:
time_importance = np.abs(attributions_np).mean(axis=2)
lead_importance = np.abs(attributions_np).mean(axis=1)
time_importance_smoothed = np.array([gaussian_filter1d(t, sigma=3) for t in time_importance])

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6))

In [ ]:
for i in range(min(10, len(time_importance_smoothed))):
    axes[0].plot(time_importance_smoothed[i], label=f'Sample {i+1}', alpha=0.7, linewidth=1.5)
axes[0].axvline(x=INPUT_LEN-50, color=COLORS['accent2'], linestyle='--', linewidth=2, label='Recent history')
axes[0].set_xlabel('Time Index (samples)')
axes[0].set_ylabel('Mean |Attribution|')
axes[0].set_title('Temporal Importance: Which Past Time Steps Matter Most?', fontweight='bold')
axes[0].legend(fontsize=8, loc='upper right')
axes[0].grid(True, alpha=0.3)

In [ ]:
mean_lead_importance = lead_importance.mean(axis=0)
std_lead_importance = lead_importance.std(axis=0)
colors_lead = plt.cm.Set3(np.linspace(0, 1, N_LEADS))
axes[1].bar(LEAD_NAMES, mean_lead_importance, yerr=std_lead_importance, capsize=4, 
           color=colors_lead, edgecolor=COLORS['text'], linewidth=1.5)
axes[1].set_ylabel('Mean |Attribution|')
axes[1].set_title('Lead Importance: Which Leads Drive Predictions?', fontweight='bold')
axes[1].grid(True, axis='y', alpha=0.3)

In [ ]:
plt.tight_layout()
save_fig('02_temporal_lead_importance.png')

In [ ]:
top_leads = np.argsort(mean_lead_importance)[-3:][::-1]
print(f"Top 3 important leads:")
for rank, lead_idx in enumerate(top_leads, 1):
    print(f"  {rank}. {LEAD_NAMES[lead_idx]} ({mean_lead_importance[lead_idx]:.4f})")

Cell 10: Sample Predictions vs Ground Truth

In [ ]:
n_show = 3
fig = plt.figure(figsize=(15, 4*n_show))
leads_to_show = [0, 1, 6, 7, 10, 11]

In [ ]:
for sample_idx in range(n_show):
    for subplot_idx, lead_idx in enumerate(leads_to_show):
        ax = plt.subplot(n_show, 6, sample_idx*6 + subplot_idx + 1)
        
        x_input = X_explain[sample_idx, :, lead_idx]
        t_input = np.arange(len(x_input))
        y_true = y_explain[sample_idx, :, lead_idx]
        t_true = np.arange(INPUT_LEN, INPUT_LEN + len(y_true))
        y_pred = predictions_np[sample_idx, :, lead_idx]
        
        ax.plot(t_input, x_input, color=COLORS['accent3'], label='Input', linewidth=1.5)
        ax.plot(t_true, y_true, color=COLORS['accent1'], label='Ground Truth', linewidth=2)
        ax.plot(t_true, y_pred, color=COLORS['accent2'], label='Prediction', linewidth=2, linestyle='--')
        ax.axvspan(INPUT_LEN, INPUT_LEN + HORIZON, alpha=0.1, color=COLORS['accent1'])
        ax.set_title(f'Lead {LEAD_NAMES[lead_idx]}', fontsize=9)
        ax.set_xlabel('Sample')
        ax.set_ylabel('ECG (norm.)')
        ax.grid(True, alpha=0.2)
        if subplot_idx == 0:
            ax.legend(fontsize=7, loc='upper left')

In [ ]:
plt.suptitle('Sample Predictions vs Ground Truth', fontsize=13, fontweight='bold')
plt.tight_layout()
save_fig('03_sample_predictions.png')

Cell 11: Occlusion Sensitivity

In [ ]:
def compute_occlusion_sensitivity(model, X_batch, y_batch, device, n_leads=12):
    model.eval()
    loss_fn = F.mse_loss
    
    with torch.no_grad():
        y_pred_baseline = model(X_batch)
        loss_baseline = loss_fn(y_pred_baseline, y_batch)
    
    occlusion_impacts = []
    for lead_idx in range(n_leads):
        X_occluded = X_batch.clone()
        X_occluded[:, :, lead_idx] = 0
        with torch.no_grad():
            y_pred_occluded = model(X_occluded)
            loss_occluded = loss_fn(y_pred_occluded, y_batch)
        occlusion_impacts.append((loss_occluded - loss_baseline).item())
    
    return np.array(occlusion_impacts)

In [ ]:
print("Computing occlusion sensitivity...")
occlusion_sensitivities = []
batch_size = 10

In [ ]:
for i in range(0, len(X_explain_torch), batch_size):
    batch_end = min(i + batch_size, len(X_explain_torch))
    sensitivity = compute_occlusion_sensitivity(model, X_explain_torch[i:batch_end], 
                                                 y_explain_torch[i:batch_end], DEVICE)
    occlusion_sensitivities.append(sensitivity)

In [ ]:
occlusion_sensitivities = np.array(occlusion_sensitivities)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

In [ ]:
mean_occlusion = occlusion_sensitivities.mean(axis=0)
std_occlusion = occlusion_sensitivities.std(axis=0)
colors_occ = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, N_LEADS))
axes[0].bar(LEAD_NAMES, mean_occlusion, yerr=std_occlusion, capsize=4, 
           color=colors_occ, edgecolor=COLORS['text'], linewidth=1.5)
axes[0].set_ylabel('Loss Increase (MSE)')
axes[0].set_title('Occlusion Sensitivity: Importance by Lead Ablation', fontweight='bold')
axes[0].grid(True, axis='y', alpha=0.3)

In [ ]:
im = axes[1].imshow(occlusion_sensitivities[:15], aspect='auto', cmap='YlOrRd')
axes[1].set_xlabel('Lead')
axes[1].set_ylabel('Sample')
axes[1].set_xticks(range(N_LEADS))
axes[1].set_xticklabels(LEAD_NAMES, fontsize=9)
axes[1].set_title('Per-Sample Occlusion Sensitivity', fontweight='bold')
plt.colorbar(im, ax=axes[1], label='Loss Increase')

In [ ]:
plt.tight_layout()
save_fig('04_occlusion_sensitivity.png')

Cell 12: Error Decomposition

In [ ]:
pred_errors = predictions_np - y_explain
rmse_per_sample = np.sqrt(np.mean(pred_errors**2, axis=(1, 2)))
mae_per_sample = np.mean(np.abs(pred_errors), axis=(1, 2))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

In [ ]:
error_by_lead = np.abs(pred_errors).mean(axis=(0, 1))
axes[0, 0].bar(LEAD_NAMES, error_by_lead, color=plt.cm.viridis(np.linspace(0, 1, N_LEADS)))
axes[0, 0].set_ylabel('Mean |Error|')
axes[0, 0].set_title('Prediction Error by Lead', fontweight='bold')
axes[0, 0].grid(True, axis='y', alpha=0.3)

In [ ]:
error_by_time = np.abs(pred_errors).mean(axis=(0, 2))
axes[0, 1].fill_between(range(HORIZON), error_by_time, alpha=0.3, color=COLORS['accent1'])
axes[0, 1].plot(error_by_time, color=COLORS['accent1'], linewidth=2)
axes[0, 1].set_xlabel('Forecast Time Step')
axes[0, 1].set_ylabel('Mean |Error|')
axes[0, 1].set_title('Prediction Error Over Forecast Horizon', fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

In [ ]:
rmse_by_lead = np.sqrt(np.mean(pred_errors**2, axis=(0, 1)))
axes[1, 0].scatter(mean_lead_importance, rmse_by_lead, s=100, alpha=0.6,
                  color=COLORS['accent1'], edgecolor=COLORS['text'], linewidth=1.5)
for i, lead in enumerate(LEAD_NAMES):
    axes[1, 0].annotate(lead, (mean_lead_importance[i], rmse_by_lead[i]), fontsize=8)
axes[1, 0].set_xlabel('Attribution Importance')
axes[1, 0].set_ylabel('RMSE')
axes[1, 0].set_title('Lead Importance vs Prediction Error', fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

In [ ]:
error_matrix = np.abs(pred_errors).mean(axis=0)
im = axes[1, 1].imshow(error_matrix.T, aspect='auto', cmap='YlOrRd')
axes[1, 1].set_xlabel('Forecast Time Step')
axes[1, 1].set_ylabel('Lead')
axes[1, 1].set_yticks(range(N_LEADS))
axes[1, 1].set_yticklabels(LEAD_NAMES, fontsize=9)
axes[1, 1].set_title('Error Heatmap: Lead × Time', fontweight='bold')
plt.colorbar(im, ax=axes[1, 1], label='|Error|')

In [ ]:
plt.tight_layout()
save_fig('05_error_analysis.png')

Cell 13: Explainability Summary

In [ ]:
fig = plt.figure(figsize=(14, 10))
gs = gridspec.GridSpec(3, 3, figure=fig)

In [ ]:
ax = fig.add_subplot(gs[0, 0])
ax.hist(attributions_np.flatten(), bins=50, color=COLORS['accent1'], edgecolor=COLORS['text'], alpha=0.7)
ax.set_xlabel('Gradient × Input')
ax.set_ylabel('Frequency')
ax.set_title('Attribution Distribution', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

In [ ]:
ax = fig.add_subplot(gs[0, 1])
colors_grad = plt.cm.Set2(np.linspace(0, 1, N_LEADS))
ax.barh(LEAD_NAMES[::-1], mean_lead_importance[::-1], color=colors_grad[::-1])
ax.set_xlabel('Mean |Attribution|')
ax.set_title('Gradient-Based Importance', fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

In [ ]:
ax = fig.add_subplot(gs[0, 2])
ax.barh(LEAD_NAMES[::-1], mean_occlusion[::-1], color=plt.cm.Oranges(np.linspace(0.4, 0.9, N_LEADS))[::-1])
ax.set_xlabel('Loss Increase')
ax.set_title('Occlusion-Based Importance', fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

In [ ]:
ax = fig.add_subplot(gs[1, 0])
ax.bar(LEAD_NAMES, rmse_by_lead, color=plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, N_LEADS)))
ax.set_ylabel('RMSE')
ax.set_title('Prediction Accuracy by Lead', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

In [ ]:
ax = fig.add_subplot(gs[1, 1:])
ax.fill_between(range(HORIZON), error_by_time, alpha=0.3, color=COLORS['accent2'])
ax.plot(error_by_time, color=COLORS['accent2'], linewidth=2)
ax.set_xlabel('Forecast Step')
ax.set_ylabel('Mean |Error|')
ax.set_title('Error Over Forecast Horizon', fontweight='bold')
ax.grid(True, alpha=0.3)

In [ ]:
ax = fig.add_subplot(gs[2, 0])
correlation = np.corrcoef(mean_lead_importance, mean_occlusion)[0, 1]
ax.scatter(mean_lead_importance, mean_occlusion, s=150, alpha=0.7,
          color=COLORS['accent3'], edgecolor=COLORS['text'], linewidth=1.5)
for i, lead in enumerate(LEAD_NAMES):
    ax.annotate(lead, (mean_lead_importance[i], mean_occlusion[i]), fontsize=8)
ax.set_xlabel('Gradient-Based Importance')
ax.set_ylabel('Occlusion Importance')
ax.set_title(f'Method Agreement (r={correlation:.3f})', fontweight='bold')
ax.grid(True, alpha=0.3)

In [ ]:
ax = fig.add_subplot(gs[2, 1:])
ax.axis('off')
insights = f"""
KEY INSIGHTS — CNN-LSTM EXPLAINABILITY

In [ ]:
📊 Model Performance:
   • Mean RMSE: {rmse_per_sample.mean():.4f}
   • Mean MAE:  {mae_per_sample.mean():.4f}

In [ ]:
🔍 Most Important Leads (Gradient):
   1. {LEAD_NAMES[top_leads[0]]:>5s}  ({mean_lead_importance[top_leads[0]]:.4f})
   2. {LEAD_NAMES[top_leads[1]]:>5s}  ({mean_lead_importance[top_leads[1]]:.4f})
   3. {LEAD_NAMES[top_leads[2]]:>5s}  ({mean_lead_importance[top_leads[2]]:.4f})

In [ ]:
⚙️ Temporal Pattern:
   • Recent samples: higher importance
   • Error increases toward horizon end

In [ ]:
✅ Method Agreement: r={correlation:.3f}
"""
ax.text(0.05, 0.95, insights, transform=ax.transAxes, fontsize=10, verticalalignment='top',
       bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))

In [ ]:
plt.suptitle('Explainability Summary Dashboard', fontsize=14, fontweight='bold')
plt.tight_layout()
save_fig('06_explainability_summary.png')

In [ ]:
print(f"\n✅ EXPLAINABILITY ANALYSIS COMPLETE")
print(f"Figures saved to: {os.path.abspath(FIG_DIR)}")